# ReAct Agents: A Bounded Tool Loop

| Field | Value |
|---|---|
| Stage | LangGraph and agentic foundations |
| Difficulty | Intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
A ReAct agent is a state machine around tool calls. Tool contracts, routing, step budgets, and terminal states matter more than hidden reasoning text.

## 30-Second Summary

This notebook builds an offline LangGraph tool loop. A deterministic planner routes an addition question to a calculator, records the observation, and finishes in one tool step; unsupported questions abstain without calling a tool.

## Why This Matters

Agent demos often hide termination and error behavior inside an LLM. Making plan, action, observation, and finish states explicit makes loops testable and prevents unbounded tool use.

## Scope

| Covers | Does not cover |
|---|---|
| Typed state, tool contract, routing, observation, abstention, step budget | Hosted LLM planner, web tools, hidden chain-of-thought, long-term memory |


## Mental Model

```text
question -> plan -> tool action -> observation -> plan -> answer/abstain
                    ^ bounded by step budget -----------|
```


In [1]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph

class AgentState(TypedDict, total=False):
    question: str
    steps: int
    action: str
    observation: str
    answer: str

MAX_TOOL_STEPS = 2

def calculator_add(left: str, right: str) -> str:
    return str(int(left) + int(right))


## How It Works

The planner emits a structured action or terminal answer. The action node validates the tool name/arguments, records an observation, increments the budget, and returns to planning. No hidden rationale is required for control flow.


## Baseline

A direct hard-coded answer would be fast but cannot expose which tool ran, enforce a common budget, or generalize to several controlled actions.


In [2]:
def direct_answer(question: str) -> str:
    return "13" if question == "What is 5 plus 8?" else "I do not know."

direct_answer("What is 5 plus 8?"), direct_answer("What is today's weather?")


('13', 'I do not know.')

## Technique Implementation

The planner is deterministic so the lesson runs offline. Replacing it with an LLM changes how actions are proposed, not the validation, execution, budget, or terminal-state responsibilities.


In [3]:
def plan(state: AgentState) -> AgentState:
    if state.get("observation"):
        return {"answer": f"The result is {state['observation']}."}
    if state.get("steps", 0) >= MAX_TOOL_STEPS:
        return {"answer": "I stopped after reaching the tool-step budget."}
    if state["question"].strip().lower() == "what is 5 plus 8?":
        return {"action": "calculator:add:5:8"}
    return {"answer": "I cannot answer with the available tools."}

def act(state: AgentState) -> AgentState:
    tool, operation, left, right = state["action"].split(":")
    if (tool, operation) != ("calculator", "add"):
        return {"answer": "Tool validation failed.", "action": ""}
    return {
        "observation": calculator_add(left, right),
        "steps": state.get("steps", 0) + 1,
        "action": "",
    }

def route(state: AgentState) -> str:
    return "finish" if state.get("answer") else "act"


## Controlled Experiment

We compile the loop, run an answerable and unsupported question, and inspect update events. Success requires one validated tool step for arithmetic and zero tool steps for abstention.


In [4]:
builder = StateGraph(AgentState)
builder.add_node("plan", plan)
builder.add_node("act", act)
builder.add_edge(START, "plan")
builder.add_conditional_edges("plan", route, {"finish": END, "act": "act"})
builder.add_edge("act", "plan")
agent = builder.compile()

answerable = agent.invoke({"question": "What is 5 plus 8?", "steps": 0})
unsupported = agent.invoke({"question": "What is today's weather?", "steps": 0})
trace = list(agent.stream({"question": "What is 5 plus 8?", "steps": 0}, stream_mode="updates"))
{"answerable": answerable, "unsupported": unsupported, "trace": trace}


{'answerable': {'question': 'What is 5 plus 8?',
  'steps': 1,
  'action': '',
  'observation': '13',
  'answer': 'The result is 13.'},
 'unsupported': {'question': "What is today's weather?",
  'steps': 0,
  'answer': 'I cannot answer with the available tools.'},
 'trace': [{'plan': {'action': 'calculator:add:5:8'}},
  {'act': {'observation': '13', 'steps': 1, 'action': ''}},
  {'plan': {'answer': 'The result is 13.'}}]}

## Evaluation

The arithmetic run follows `plan → act → plan`, calls one tool, observes `13`, and terminates. The weather question abstains in the first planning node with zero tool calls. The trace exposes control state without storing private chain-of-thought.


In [5]:
assert answerable["answer"] == "The result is 13." and answerable["steps"] == 1
assert unsupported["answer"] == "I cannot answer with the available tools." and unsupported["steps"] == 0
assert [next(iter(event)) for event in trace] == ["plan", "act", "plan"]
assert trace[1]["act"]["observation"] == "13"
print("Bounded ReAct checks passed.")


Bounded ReAct checks passed.


## Decision Guide

| Need | Architecture |
|---|---|
| Fixed known sequence | Deterministic chain/graph |
| One bounded tool choice | Single ReAct loop |
| Complex dependencies | Explicit workflow planner |
| No authorized supporting tool | Abstain, do not improvise |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Infinite loop | No terminal condition/budget | Step/time/tool budgets |
| Wrong tool arguments | Free-form action | Structured schema and validation |
| Tool error becomes answer | Observation unchecked | Typed error branch/retry policy |
| Conversation leaks across users | Memory scope wrong | Explicit thread/tenant boundary |


## Production Notes

### Observability
Trace node, action schema, safe arguments, tool result status, step count, latency, terminal reason, and model/version.

### Safety and Guardrails
Allowlist tools, validate arguments, authorize every call, and require confirmation for consequential actions.

### Latency and Cost
Each loop step may add model and tool calls; enforce budgets and prefer deterministic workflows when the route is known.


## Practice

Add a divide tool with zero-division handling and prove that the loop terminates with a typed error observation.

## Recall

Toggle - Recall: What makes the loop safe to operate?
Structured actions, validation, authorization, budgets, and terminal states.

Toggle - Recall: Why avoid storing hidden reasoning?
Control flow can be observed through actions and state without collecting sensitive internal rationale.

## Sources

- [ReAct paper](https://arxiv.org/abs/2210.03629)
- [LangGraph workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents)

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the deterministic bounded loop | Add typed tool errors, retries, and checkpoint boundaries |
